In [36]:
import pandas as pd
import numpy as np
from BK_Tree import BKTree
import re
import string
from dead_processing_utils import *
from file_processing_utils import *

In [37]:
folder_path = 'dead_records_data'
full_data = concat_df(folder_path)
# full_data.to_csv('data/all_dead_records.csv', index=False)
print(len(full_data))

21404


In [38]:
in_db_taxon = pd.read_csv('output/final_taxon_key.csv')
in_db_taxon = in_db_taxon[['deadcard_taxon', 'db_taxon']]

out_db_taxon = pd.read_csv('output/to_add_final.csv')
out_db_taxon = out_db_taxon[['name', 'checked_synonym']]
out_db_taxon = out_db_taxon.rename(columns={'name': 'deadcard_taxon', 'checked_synonym': 'db_taxon'})

missed_taxon = pd.read_csv('output/solved_taxon_key_additions.csv')
missed_taxon = missed_taxon[['name', 'checked_synonym']]
missed_taxon = missed_taxon.rename(columns={'name': 'deadcard_taxon', 'checked_synonym': 'db_taxon'})

taxon = pd.concat([in_db_taxon, out_db_taxon, missed_taxon], axis=0, ignore_index=True)

taxon.to_csv('output/final_final_taxon_key.csv', index=False)

In [39]:
taxon = pd.read_csv('output/final_final_taxon_key.csv')

In [40]:
collectors = pd.read_csv('output/collector_name_matching_gpt_EDITED.csv')
collector_key = collectors.set_index('original_name')['matched_name'].to_dict()

taxon_key = taxon.set_index('deadcard_taxon')['db_taxon'].to_dict()

places = pd.read_csv('output/unique_locations_EDITED.csv')
places_key = places.set_index('0')['***DbPlaceGuess'].to_dict()

seperating duplicate accessions from non-duplication accessions based on first instance

In [41]:
full_data_no_dupes = full_data[~full_data['objectNumber'].str.split('_').apply(lambda x: 'DUPLICATE' in x)]
full_data_dupes = full_data[full_data['objectNumber'].str.split('_').apply(lambda x: 'DUPLICATE' in x)]

print(len(full_data_no_dupes) + len(full_data_dupes))
print(len(full_data_no_dupes))
print(len(full_data_dupes))

21404
20592
812


In [42]:
def data_formatter(date):
    date = str(date)
    if not any('-' in chr for chr in date):
        if len(date.split(' ')) == 2:
            return date.split(' ')[1]
    elif not date.split('-')[0].isdigit():
        return '1-' + date
    else:
        return date

In [43]:
object_upload = strip('ucbg_templates/ucbg_4-0-0_collectionobject-template.csv')

object_upload['objectNumber'] = full_data_no_dupes['objectNumber']

object_upload['collector'] = full_data_no_dupes['fieldCollector'].apply(lambda x: collector_key.get(x, ''))
object_upload['taxon'] = full_data_no_dupes['taxon'].apply(lambda x: taxon_key.get(x, ''))
object_upload['fieldLocPlace'] = full_data_no_dupes['fieldLocCountry'].apply(lambda x: places_key.get(x, ''))

object_upload['fieldCollectionDateGroup'] = full_data_no_dupes['fieldCollectionDateGroup'].apply(data_formatter)

object_upload['accessionDate'] = full_data_no_dupes['accessionDate']
object_upload['briefDescription'] = full_data_no_dupes['briefDescription']
object_upload['deadFlag'] = full_data_no_dupes['deadFlag']

object_upload['comment'] = full_data_no_dupes['comment']


object_upload.to_csv('output/object_upload.csv', index=False)

print((object_upload['taxon'] == '').sum())

497
